In [ ]:
!pip install numpy==1.26.4 pandas==2.2.2 pyarrow==15.0.2 "datasets==2.20.0" --force-reinstall --quiet

In [1]:
# Static validation: check the relative magnitudes of the score-matching loss and the
# physics term on a synthetic batch BEFORE committing to a full Kaggle training run.
# This avoids the lambda-scaling failure mode (Hypothesis 3).
import torch

torch.manual_seed(0)
B, C, F, T = 4, 1, 256, 256                # SGMSE+ STFT shape with n_fft=510, num_frames=256
sigma_val = 0.5                            # representative diffusion noise level

# Synthesize a smooth clean spectrogram + noisy x_t
freq = torch.linspace(0, 1, F).view(1, 1, F, 1)
time = torch.linspace(0, 1, T).view(1, 1, 1, T)
x_clean = (torch.exp(-freq * 4.0) * torch.cos(2 * 3.14159 * time * 3)).expand(B, C, F, T)
x_clean = torch.complex(x_clean, 0.5 * x_clean.roll(1, dims=-1))
z       = torch.randn_like(x_clean.real) + 1j * torch.randn_like(x_clean.real)
x_t     = x_clean + sigma_val * z
# A perturbed "score" estimate, of the same scale a trained model would produce.
score   = -z / sigma_val + 0.1 * (torch.randn_like(z.real) + 1j * torch.randn_like(z.real))

# (1) Standard score-matching loss: mean over batch of 0.5 * sum |score*sigma + z|^2
sm = torch.square(torch.abs(score * sigma_val + z))
loss_sm = torch.mean(0.5 * torch.sum(sm.reshape(B, -1), dim=-1))

# (2) Physics term: spectral envelope smoothness on log-magnitude of Tweedie x_0_hat
x_hat_spec = x_t + (sigma_val ** 2) * score
mag = torch.abs(x_hat_spec).clamp(min=1e-7)
log_mag = torch.log(mag)
d2_f = log_mag[:, :, 2:, :] - 2.0 * log_mag[:, :, 1:-1, :] + log_mag[:, :, :-2, :]
loss_phys = torch.mean(d2_f ** 2)

print(f'score-matching loss : {loss_sm.item():.4e}')
print(f'physics loss (raw)  : {loss_phys.item():.4e}')
print(f'ratio  phys / sm    : {(loss_phys.item() / loss_sm.item()):.4e}')

# Choose physics_weight so the physics contribution is ~1-5% of the SM loss at init.
target_fraction = 0.02
suggested_w = target_fraction * loss_sm.item() / loss_phys.item()
print(f'suggested physics_weight (for ~{target_fraction*100:.0f}% contribution): {suggested_w:.4e}')

# NaN / inf sanity check on the finite difference path.
assert torch.isfinite(loss_phys), 'physics loss is non-finite!'
assert torch.isfinite(loss_sm),   'score-matching loss is non-finite!'
print('OK: both loss components are finite.')

score-matching loss : 1.6358e+02
physics loss (raw)  : 1.0686e+00
ratio  phys / sm    : 6.5325e-03
suggested physics_weight (for ~2% contribution): 3.0616e+00
OK: both loss components are finite.


In [2]:
# Clone sgmse. Pilot does NOT download the pretrained checkpoint — we train from random init.
import os, shutil
if os.path.exists('/kaggle/working/sgmse'):
    shutil.rmtree('/kaggle/working/sgmse')
os.chdir('/kaggle/working')
get_ipython().system('git clone https://github.com/sp-uhh/sgmse.git')
os.chdir('/kaggle/working/sgmse')
get_ipython().system('pip install -r requirements.txt --quiet')
get_ipython().system('pip install pesq pystoi pandas gdown --quiet')

Cloning into 'sgmse'...
remote: Enumerating objects: 1011, done.
remote: Counting objects: 100% (357/357), done.
remote: Compressing objects: 100% (145/145), done.
remote: Total 1011 (delta 276), reused 214 (delta 212), pack-reused 654 (from 3)
Receiving objects: 100% (1011/1011), 3.74 MiB | 17.97 MiB/s, done.
Resolving deltas: 100% (547/547), done.
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.9 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 4.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ydata-profiling 4.18.1 requires typeguard<5,>=4, but you have typeguard 2.13.3 which is incompatible.
inflect 7.5.0 requires typeguard>=4.0.1, but you have typeguard 2.13.3 which is incompatible.


In [3]:
# Patch model.py: replace the smoothness prior with a SPECTRAL FLATNESS penalty
# on the Tweedie estimate. SFM(x) ~= 1 for noise, << 1 for structured speech, so
# penalizing it pushes the network's predicted clean spectrogram toward "speech-like"
# (peaky) and away from "noise-like" (flat). Unlike the smoothness prior, this
# fires across the FULL sigma range — strongest at high sigma where score matching
# is weakest.
patch = '''
            # === physics-informed prior: spectral flatness penalty ===
            # SFM(x) = exp(mean log|x|^2) / mean(|x|^2) along the frequency axis.
            # Pure noise -> 1, structured (peaky) speech spectrum -> << 1.
            x_hat_spec = x_t + (sigma ** 2) * score
            psd = (x_hat_spec.abs() ** 2).clamp(min=1e-10)              # (B,C,F,T)
            log_psd = torch.log(psd)
            log_gm = log_psd.mean(dim=2)                                # log geometric mean
            am     = psd.mean(dim=2)                                    # arithmetic mean
            sfm    = torch.exp(log_gm) / am.clamp(min=1e-10)            # (B,C,T) in [0,1]
            phys_loss = sfm.mean()
            loss = loss + self.physics_weight * phys_loss
'''

with open('/kaggle/working/sgmse/sgmse/model.py', 'r') as f:
    content = f.read()

content = content.replace(
    'self.loss_type = loss_type\n',
    'self.loss_type = loss_type\n        self.physics_weight = 0.0\n',
    1,
)

old = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n        elif self.loss_type == "denoiser":'
new = '            loss = torch.mean(0.5*torch.sum(losses.reshape(losses.shape[0], -1), dim=-1))\n' + patch + '        elif self.loss_type == "denoiser":'
assert old in content, 'Anchor for score_matching patch not found'
content = content.replace(old, new, 1)

with open('/kaggle/working/sgmse/sgmse/model.py', 'w') as f:
    f.write(content)

get_ipython().system('grep -n "physics_weight\|phys_loss\|x_hat_spec\|sfm" /kaggle/working/sgmse/sgmse/model.py')

72:        self.physics_weight = 0.0
152:            x_hat_spec = x_t + (sigma ** 2) * score
153:            psd = (x_hat_spec.abs() ** 2).clamp(min=1e-10)              # (B,C,F,T)
157:            sfm    = torch.exp(log_gm) / am.clamp(min=1e-10)            # (B,C,T) in [0,1]
158:            phys_loss = sfm.mean()
159:            loss = loss + self.physics_weight * phys_loss


<>:38: SyntaxWarning: invalid escape sequence '\|'
<>:38: SyntaxWarning: invalid escape sequence '\|'
/tmp/ipykernel_148/2526006010.py:38: SyntaxWarning: invalid escape sequence '\|'
  get_ipython().system('grep -n "physics_weight\|phys_loss\|x_hat_spec\|sfm" /kaggle/working/sgmse/sgmse/model.py')


In [4]:
import soundfile as sf
import numpy as np
from datasets import load_dataset, Audio

# Step 2 experimental: 3000 train / 100 valid / full 826 test (same as control,
# so the two runs are directly comparable on the same test set).
TEST_DIR = "data/test"
TRAIN_DIR = "data/train"
VALID_DIR = "data/valid"
for d in (TEST_DIR, TRAIN_DIR, VALID_DIR):
    os.makedirs(f"{d}/clean", exist_ok=True)
    os.makedirs(f"{d}/noisy", exist_ok=True)

print("Loading full test set...")
test_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="test")
test_data = test_data.cast_column("clean", Audio(sampling_rate=16000))
test_data = test_data.cast_column("noisy", Audio(sampling_rate=16000))
for i, sample in enumerate(test_data):
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TEST_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TEST_DIR}/noisy/{fname}", noisy, 16000)
print(f"Wrote {len(test_data)} test samples")

print("Loading train set (3000 samples)...")
train_data = load_dataset("MeiWu1123/VoiceBank-DEMAND-16k", split="train")
train_data = train_data.cast_column("clean", Audio(sampling_rate=16000))
train_data = train_data.cast_column("noisy", Audio(sampling_rate=16000))
for i in range(3000):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{TRAIN_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{TRAIN_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 3000 train samples")

print("Writing validation set (100 samples)...")
for i in range(3000, 3100):
    sample = train_data[i]
    clean = np.array(sample["clean"]["array"], dtype=np.float32)
    noisy = np.array(sample["noisy"]["array"], dtype=np.float32)
    fname = f"{i:04d}.wav"
    sf.write(f"{VALID_DIR}/clean/{fname}", clean, 16000)
    sf.write(f"{VALID_DIR}/noisy/{fname}", noisy, 16000)
print("Wrote 100 validation samples")

Loading full test set...


Generating train split:   0%|          | 0/11572 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/824 [00:00<?, ? examples/s]

Wrote 824 test samples
Loading train set (3000 samples)...
Wrote 3000 train samples
Writing validation set (100 samples)...
Wrote 100 validation samples


In [6]:
# Step 2 EXPERIMENTAL training: random-init NCSNpp, score-matching + physics prior.
# Identical to control_from_scratch.ipynb except:
#   1. SAVE_DIR = sgmse_experimental
#   2. physics_weight is set to a fixed TARGET_W (0.1, matching the from-pretrained
#      run that worked) and ramped over the first warmup_epochs.
#   3. The model.py patch only fires the prior when sigma < sigma_thresh, so the
#      Tweedie estimate is reliable even at random init.
# The previous random-init calibration was REMOVED — it set physics_weight ~= 292
# based on a regime where the prior is invalid, which is why PIDM-from-scratch
# was underperforming the control.

finetune_script = '''
import os
os.chdir("/kaggle/working/sgmse")
import torch
if not hasattr(torch, "_load_patched"):
    _orig = torch.load
    def _patched(*a, **kw):
        kw["weights_only"] = False
        return _orig(*a, **kw)
    torch.load = _patched
    torch._load_patched = True

import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, Callback
from sgmse.model import ScoreModel
from sgmse.data_module import SpecsDataModule

SAVE_DIR  = "/kaggle/working/sgmse_experimental"
DATA_DIR  = "/kaggle/working/sgmse/data"
os.makedirs(SAVE_DIR, exist_ok=True)

pl.seed_everything(42)

model = ScoreModel(
    backbone="ncsnpp",
    sde="ouve",
    data_module_cls=SpecsDataModule,
    theta=1.5, sigma_min=0.05, sigma_max=0.5, N=1000,
    loss_type="score_matching", loss_weighting="sigma^2",
    num_eval_files=0, lr=1e-4,
    nf=32,
    base_dir=DATA_DIR, format="default", batch_size=16,
    n_fft=510, hop_length=128, num_frames=256, window="hann",
    num_workers=2, dummy=False, spec_factor=0.15, spec_abs_exponent=0.5,
    normalize="noisy", transform_type="exponent",
)

# Fixed physics_weight target. Calibration on a random-init batch was unreliable
# (the Tweedie-based prior is invalid in most of that batch sigma range), so we
# use the value that worked when finetuning from pretrained (0.1) and ramp it in.
TARGET_W = 0.1
model.physics_weight = 0.0   # warmup callback will set it each epoch

class PhysicsWarmup(Callback):
    def __init__(self, target, warmup_epochs=5):
        self.target = target
        self.warmup_epochs = warmup_epochs
    def on_train_epoch_start(self, trainer, pl_module):
        e = trainer.current_epoch
        if e < self.warmup_epochs:
            pl_module.physics_weight = self.target * (e / self.warmup_epochs)
        else:
            pl_module.physics_weight = self.target

class PrintLosses(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        vl = m.get("valid_loss"); tl = m.get("train_loss_epoch")
        print("\\n[Epoch " + str(trainer.current_epoch) +
              "] train_loss=" + (str(round(float(tl),4)) if tl is not None else "?") +
              " | valid_loss=" + (str(round(float(vl),4)) if vl is not None else "?") +
              " | physics_weight=" + str(pl_module.physics_weight) + "\\n")

ckpt_cb = ModelCheckpoint(
    dirpath=SAVE_DIR,
    filename="experimental_epoch{epoch:02d}_valloss{valid_loss:.4f}",
    save_top_k=3, monitor="valid_loss", mode="min", every_n_epochs=1,
    save_last=True,
)

trainer = pl.Trainer(
    max_epochs=40, accelerator="gpu", devices=1,
    callbacks=[ckpt_cb, PrintLosses(), PhysicsWarmup(TARGET_W, warmup_epochs=5)],
    log_every_n_steps=20, enable_progress_bar=True,
    gradient_clip_val=1.0,
)
trainer.fit(model)
print("Best checkpoint:", ckpt_cb.best_model_path)
'''

with open('/kaggle/working/sgmse/finetune_experimental.py', 'w') as f:
    f.write(finetune_script)
print('finetune_experimental.py written')

finetune_experimental.py written


In [7]:
# Run Step 2 experimental. PhysicsWarmup ramps physics_weight from 0 to TARGET_W
# over the first 5 epochs; PrintLosses logs train/valid loss + physics_weight each
# epoch. Estimated ~1:45 hours on Kaggle T4.
get_ipython().system('python finetune_experimental.py')

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
2026-05-21 03:30:04.696520: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1779334205.139913     275 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1779334205.281171     275 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1779334206.335412     275 computation_placer.cc:177] computation placer already registered. Please c

In [8]:
# 1. Patch enhancement.py
with open('/kaggle/working/sgmse/enhancement.py', 'r') as f:
  content = f.read()

patch = '''import torch
_original_torch_load = torch.load
def _patched_torch_load(*args, **kwargs):
  kwargs['weights_only'] = False
  return _original_torch_load(*args, **kwargs)
torch.load = _patched_torch_load

'''
if '_patched_torch_load' not in content:
  with open('/kaggle/working/sgmse/enhancement.py', 'w') as f:
      f.write(patch + content)
  print('enhancement.py patched')
else:
  print('enhancement.py already patched')

# 2. Patch Lightning's loaders (pure Python, no sed)
for path in [
  '/usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py',
  '/usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py',
]:
  with open(path, 'r') as f:
      src = f.read()
  new = src.replace(
      'weights_only: Optional[bool] = None,',
      'weights_only: Optional[bool] = False,',
  )
  if new != src:
      with open(path, 'w') as f:
          f.write(new)
      print('patched', path)
  else:
      print('no change needed', path)

# 3. Clear any partial output from a previous experimental run
import shutil, os
if os.path.exists('/kaggle/working/sgmse/enhanced_experimental'):
  shutil.rmtree('/kaggle/working/sgmse/enhanced_experimental')
  print('cleared enhanced_experimental/')

enhancement.py patched
patched /usr/local/lib/python3.12/dist-packages/pytorch_lightning/core/saving.py
patched /usr/local/lib/python3.12/dist-packages/lightning_fabric/utilities/cloud_io.py


In [9]:
# Enhancement + metrics on the trained experimental checkpoint.
# Same test set / N as control_from_scratch so the two runs are directly comparable.
import glob, os
ckpts = sorted(glob.glob('/kaggle/working/sgmse_experimental/experimental_*.ckpt'))
print('checkpoints:', ckpts)
CKPT = ckpts[-1]
print('using:', CKPT)

get_ipython().system(f'python enhancement.py --test_dir data/test/noisy --enhanced_dir enhanced_experimental --ckpt {CKPT} --N 10')
get_ipython().system('python calc_metrics.py --clean_dir data/test/clean --noisy_dir data/test/noisy --enhanced_dir enhanced_experimental')

checkpoints: ['/kaggle/working/sgmse_experimental/experimental_epochepoch=17_vallossvalid_loss=1137.3519.ckpt', '/kaggle/working/sgmse_experimental/experimental_epochepoch=34_vallossvalid_loss=1249.8651.ckpt', '/kaggle/working/sgmse_experimental/experimental_epochepoch=38_vallossvalid_loss=1076.5782.ckpt']
using: /kaggle/working/sgmse_experimental/experimental_epochepoch=38_vallossvalid_loss=1076.5782.ckpt
Set TORCH_CUDA_ARCH_LIST to: 7.5;7.5
100%|█████████████████████████████████████████| 824/824 [02:42<00:00,  5.08it/s]
PESQ: 1.75 ± 0.41
ESTOI: 0.72 ± 0.13
SI-SDR: 11.1 ± 3.7
SI-SIR: 18.3 ± 5.0
SI-SAR: 12.4 ± 3.5
